# Reranking the hybrid's candidates: MaxP windows and blending (Colab GPU)

Re-tests cross-encoder reranking with two fixes for why the first attempt lowered Recall@10
(0.722 → 0.637 and 0.669, see `docs/FULL_CORPUS.md`):

1. **MaxP:** each candidate is scored on several windows across its **whole document**, each
   starting with the title, and keeps its best window's score. Before, it was scored on one
   3,000-character slice cut to 512 tokens, which often missed the answer in long documents.
2. **Blending:** the reranker's rank is combined with the hybrid rank (weighted RRF) instead of
   replacing it, so BM25+'s exact matches on codenames and IDs are kept. Each question's
   candidates are scored once and reused for every blend weight.

The blend weight is chosen by **5-fold cross-validation on the 300 development questions**:
each question's result uses the weight that did best on the other folds, so the
cross-validated (`blendcv`) score is not tuned on the questions it reports.

1. **Runtime → Change runtime type → L4 (or T4) GPU, with High-RAM on.**
2. It reuses files on Google Drive: `documents.parquet`, `questions.jsonl`, the BM25+ index and
   the BGE-base index. No new embedding is needed.
3. **Part 3** scores the settings on development (roughly 10–30 minutes per setting).
   **Part 4** compares them with the current best, the hybrid without a reranker (Recall@10
   0.722), with 95% bootstrap intervals. **Part 5** runs one calibration check only if the
   cross-validated result is clearly better. The final split is never used.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

## 1. Drive and project code

Set `WORK` to the Drive folder that holds the data. The defaults match the personal `rag-dense`
layout (`dense_base_titled` at its top level). For the team `files_web_app` folder, use
`DENSE = f"{WORK}/indexes/dense-bge-base"`.

In [ ]:
import json
import os
from google.colab import drive

drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/rag-dense"
CORPUS = f"{WORK}/documents.parquet"
QUESTIONS = f"{WORK}/questions.jsonl"
DRIVE_BM25 = f"{WORK}/indexes/bm25"
DENSE = f"{WORK}/dense_base_titled"  # BGE-base, title on every passage
BM25 = "/content/indexes/bm25"  # Local copy: much faster than reading from Drive.

REPO = "https://github.com/AgenticEnterpriseSystem/Agentic-Enterprise-RAG-System.git"
BRANCH = "hira/dev"
# For a private repository, clone with a GitHub token instead (it is not displayed):
# from getpass import getpass; REPO = f"https://{getpass('GitHub token: ')}@github.com/AgenticEnterpriseSystem/Agentic-Enterprise-RAG-System.git"
if not os.path.exists("/content/project"):
    !git clone --depth 1 -b {BRANCH} {REPO} /content/project
%cd /content/project
!git log -1 --format='%h %s'
!pip install -q -r requirements.txt

## 2. Check the indexes

Both indexes must come from the same `documents.parquet`. If the BM25+ index on Drive is missing
or was built from a different copy, it is rebuilt here (about 5 minutes).

In [ ]:
for path in (CORPUS, QUESTIONS, f"{DENSE}/manifest.json"):
    assert os.path.exists(path), f"Missing {path}"
dense_manifest = json.load(open(f"{DENSE}/manifest.json"))
assert dense_manifest.get("complete"), "The dense index is incomplete"
from src.retrieval.corpus_dense import model_label
LABEL = model_label(dense_manifest["model"]["name"])
print("Dense model:", LABEL, dense_manifest["model"]["revision"])

def bm25_matches(path):
    manifest = f"{path}/manifest.json"
    return os.path.exists(manifest) and json.load(open(manifest))["corpus_sha256"] == dense_manifest["corpus_sha256"]

if not bm25_matches(BM25):
    if bm25_matches(DRIVE_BM25):
        print("Copying the BM25+ index from Drive...")
        !rm -rf {BM25} && mkdir -p /content/indexes && cp -r {DRIVE_BM25} {BM25}
    else:
        print("Building a BM25+ index from the same corpus (about 5 minutes)...")
        !python -m src.retrieval.corpus_bm25 --corpus {CORPUS} --output {BM25}
assert bm25_matches(BM25)
print("Indexes ready.")

## 3. Development runs

Each setting saves one run per blend weight (`blend1` = reranker order only, as before) plus a
cross-validated run (`blendcv`). Settings:

- **A:** 2,000-character windows (about 500 tokens) at a 512-token limit, up to 8 windows per
  document (the lead plus the windows that cover the most question words).
- **B:** 4,000-character windows at a 1,024-token limit: fewer, longer windows. Slower.

The hybrid itself is unchanged: BM25+ weight 0.7, 50 candidates per retriever, RRF k = 60.

In [ ]:
RERANKER = "bge-reranker-v2-m3"
BLEND = "1,0.7,0.5,0.3,0.2,0.1"
SETTINGS = {"A": (2000, 512), "B": (4000, 1024)}  # window characters, token limit
RUN = ["A", "B"]  # Remove "B" to save time.

COMMON = (f"--bm25-index {BM25} --dense-index {DENSE} --corpus {CORPUS} --questions {QUESTIONS} "
          f"--device cuda --bm25-weights 0.7 --rerank-bm25-weight 0.7 --rerankers {RERANKER} "
          f"--rerank-weights {BLEND} --rerank-max-windows 8 --reranker-batch-size 32")
for name in RUN:
    chars, tokens = SETTINGS[name]
    print(f"\n=== Setting {name}: {chars}-character windows, {tokens} tokens ===")
    !python -m src.evaluation.run_hybrid {COMMON} --split development \
        --rerank-window-chars {chars} --reranker-max-length {tokens}

## 4. Compare with the current best

Only the cross-validated (`blendcv`) runs are a fair estimate: the fixed-weight runs show which
weights work, but picking the best of them on the same questions overstates it.

In [ ]:
import glob
from src.evaluation.retrieval_metrics import evaluate_retrieval, ndcg_at_k, paired_bootstrap, recall_at_k

def load_rows(path):
    return [json.loads(line) for line in open(f"{path}/predictions.jsonl")]

BASE = "evaluation/results/hybrid_bge-base_w0.7_full_development"  # Current best, in the repo.
base = load_rows(BASE)
o = evaluate_retrieval(base)["overall"]
print(f"{'run':72} {'R@10':>6} {'nDCG@10':>8}   R@10 change (95% interval)")
print(f"{'hybrid, no reranker (current best)':72} {o['recall@10']['mean']:>6.3f} {o['ndcg@10']['mean']:>8.3f}")
cv_runs = {}
for path in sorted(glob.glob(f"evaluation/results/hybrid_{LABEL}_w0.7_{RERANKER}_maxp*_full_development")):
    rows, name = load_rows(path), os.path.basename(path)
    o = evaluate_retrieval(rows)["overall"]
    r = paired_bootstrap(rows, base, metric=recall_at_k)
    print(f"{name:72} {o['recall@10']['mean']:>6.3f} {o['ndcg@10']['mean']:>8.3f}   "
          f"{r['change']:+.3f} ({r['low']:+.3f} to {r['high']:+.3f})")
    if "_blendcv_" in name:
        cv_runs[name] = (o["recall@10"]["mean"], o["ndcg@10"]["mean"], r, path)

best_cv = max(cv_runs, key=lambda n: cv_runs[n][:2])
recall, ndcg, change, path = cv_runs[best_cv]
summary = json.load(open(f"{path}/summary.json"))
reranker = summary["index"]["reranker"]
WINDOW_CHARS, MAX_LENGTH = reranker["windows"]["chars"], reranker["max_length"]
print(f"\nBest cross-validated setting: {best_cv}")
print("Weights chosen per fold:", summary["cross_validation"]["chosen_per_fold"])

# One weight for calibration: the best fixed weight of that setting on all development questions.
stem = path.replace("_blendcv_full_development", "")
fixed = {}
for p in glob.glob(f"{stem}*_full_development"):
    if "_blendcv_" in p:
        continue
    tail = p[len(stem):].replace("_full_development", "")
    weight = float(tail.removeprefix("_blend")) if tail else 1.0
    o = evaluate_retrieval(load_rows(p))["overall"]
    fixed[weight] = (o["recall@10"]["mean"], o["ndcg@10"]["mean"], -weight)
RERANK_WEIGHT = max(fixed, key=lambda w: fixed[w])
print(f"Weight for the calibration check: {RERANK_WEIGHT:g}")
RUN_CALIBRATION = change["low"] > 0  # Better than the current best, interval above zero.
print("Run the calibration check:", RUN_CALIBRATION)

## 5. Calibration: one check of the chosen setting

Runs only if the cross-validated development result is better than the current best with a 95%
interval above zero. It scores the chosen setting and the hybrid without a reranker on the 100
calibration questions. Calibration has been used once already, so run this **once** and do not
change settings based on it.

In [ ]:
CALIBRATION = (COMMON.replace(f"--rerank-weights {BLEND}", f"--rerank-weights {RERANK_WEIGHT:g}")
               + f" --split calibration --rerank-window-chars {WINDOW_CHARS} --reranker-max-length {MAX_LENGTH}")
if RUN_CALIBRATION:
    !python -m src.evaluation.run_hybrid {CALIBRATION}
else:
    print("Skipped: the cross-validated result was not clearly better on development.")

## 6. Save the results

Copies the result folders to Drive (`results/` in `WORK`) and downloads
`reranker_results.zip`. Upload the zip to add the results to the repository.

In [ ]:
KEEP = f"hybrid_{LABEL}_w0.7_{RERANKER}_maxp* hybrid_{LABEL}_w0.7_full_calibration"
!mkdir -p {WORK}/results
!cd evaluation/results && cp -r {KEEP} {WORK}/results/ 2>/dev/null; true
!cd evaluation/results && zip -qr /content/reranker_results.zip {KEEP} 2>/dev/null; true
from google.colab import files
files.download("/content/reranker_results.zip")